# Editing models: every list of MODEL_LIST.md on the dev samples of its task

Every model of MODEL_LIST.md on a small fixed set of dev samples of the final dataset (`cryandrrich/editjudge`):
1. **Samples**: 30 dev samples per task, chosen by their request categories (`python -m src.editors samples`); a
   sample belongs to one task only. Tasks: remove, add_replace (lists 2, 3), enhance, restore, change_color,
   formatting (lists 4 to 7) and open_ended (list 1 only).
2. **Plan** (remove, add_replace, change_color, formatting): Qwen2.5-VL-7B reads each request: what to find and
   paint (remove, replace, add, recolor), or the format action (crop, rotate, resize, extend, ...).
3. **Masks** (remove, add_replace, recolor): Grounding DINO + SAM 2.1. Outputs of mask models keep only the masked
   region.
4. **Run** every model, each in its own process (and its own environment when it pins other packages). Outputs,
   seconds per image and peak GPU memory go to `out/editors/<model>/`. The generalists of list 1 run on every task.
5. **Score** with the scorer (`mlp-siglip`) next to the human edits, **report** per task
   (`out/editors/<task>/report.md` and contact sheets: team only, they show real people) and the **summary** of
   every task and model (`out/editors/summary.md`).

`SESSION` picks what one Kaggle session runs (cell 3): "specialists" (lists 2 to 7) or "shared" (list 1, the
generalists, slow: two sessions). `SMOKE_MODELS` lists the models that still need their smoke run (5 samples per
task: does it load in fp16, fit a T4, give no black images, how long does it take); the others run on all 30
samples, after the coordinator approves.

How to run:
1. Inputs: the dataset `cryandrrich/editjudge`; for the scores, also the output of `scorer_train` (it holds
   `out/scorer/runs/mlp-siglip/head.pt`); and every earlier output of this notebook (plans, masks and outputs are
   reused, so a session goes on where the last one stopped). Settings: GPU T4 x2, internet on. Push the code first.
2. Secret `HF_TOKEN` (Add-ons > Secrets), from an account that accepted the licence of
   black-forest-labs/flux-3-image-klein (needed for `flux3-klein`; faster downloads for the rest).
3. **Save Version > Save & Run All (Commit)**.

Disk: weights go to /tmp (/kaggle/working holds 20 GB and is the saved output); each model's weights are deleted
after its run. The largest downloads: OmniGen2 31 GB, VAREdit 22 GB, FLUX 3 Klein 20 GB, FLUX.2 klein 16 GB,
InstantIR 15 GB.

In [ ]:
import glob, json, os, shutil, subprocess, sys

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "curr"
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

def find_dataset():
    """The input folder that holds pairs.jsonl and images/ (the final dataset)."""
    for pattern in ("/kaggle/input/*/pairs.jsonl", "/kaggle/input/*/*/pairs.jsonl", "/kaggle/input/*/*/*/pairs.jsonl"):
        found = sorted(glob.glob(pattern))
        if found:
            return os.path.dirname(found[0])
    raise SystemExit("no input with pairs.jsonl: add the dataset cryandrrich/editjudge")

def find_input_folders(tail, max_depth=6):
    """Folders under /kaggle/input whose path ends with `tail` (never inside a dataset's images/ folder)."""
    found = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        if root.endswith("/" + tail):
            found.append(root)
            dirs[:] = []
        elif root.count("/") - 2 >= max_depth:
            dirs[:] = []
        else:
            dirs[:] = [name for name in dirs if name != "images"]
    return found

DATA = find_dataset()
os.environ["EJ_DATA_DIR"] = DATA            # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT           # all results are written here
os.environ["EJ_ENVS_DIR"] = "/tmp/ej-envs"
os.makedirs(f"{OUT}/editors", exist_ok=True)
os.chdir(CODE)

# an earlier output of this notebook (to go on where it stopped), and the scorer's trained heads
for folder in find_input_folders("out/editors"):
    shutil.copytree(folder, f"{OUT}/editors", dirs_exist_ok=True)
    print("earlier editors output copied from", folder)
runs = [f for f in find_input_folders("scorer/runs") if glob.glob(f"{f}/mlp-siglip/head.pt")]
if runs:
    shutil.copytree(runs[0], f"{OUT}/scorer/runs", dirs_exist_ok=True)
    print("scorer heads copied from", runs[0])
else:
    print("no scorer_train output among the inputs: the scoring step is skipped")

# Model weights go to /tmp: /kaggle/working is limited to 20 GB and is saved as the notebook's output
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["EJ_MODELS_DIR"] = "/tmp/models"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    os.environ["HF_HUB_VERBOSITY"] = "error"

In [ ]:
!pip -q install -U "transformers>=4.56,<5" "diffusers>=0.35,<0.40" "peft>=0.14" accelerate sentencepiece opencv-python-headless
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run_editors(*args, check=True):
    """python -m src.editors <args>: prints every line as it comes."""
    process = subprocess.Popen([sys.executable, "-u", "-m", "src.editors", *map(str, args)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() and check:
        raise subprocess.CalledProcessError(process.returncode, args)
    return process.returncode

def drop_cache(*repos):
    """Delete downloaded weights that are no longer needed (the disk holds about 70 GB)."""
    for repo in repos:
        shutil.rmtree(f"/tmp/hf/hub/models--{repo.replace('/', '--')}", ignore_errors=True)
    print(f"{shutil.disk_usage('/tmp').free / 1e9:.0f} GB free")

run_editors("list")

In [ ]:
# What this session runs. "specialists": lists 2 to 7; "shared": list 1 (the generalists, on every task)
SESSION = "specialists"
SPECIALISTS = {
    "remove": ["lama", "migan", "rorem", "objectclear", "osor"],
    "add_replace": ["sdxl-inpaint", "paint-by-inpaint", "erasedraw"],
    "enhance": ["realesrgan", "hvi-cidnet", "instructir", "vosr"],
    "restore": ["vosr", "instantir", "codeformer", "gfpgan"],
    "change_color": ["ddcolor", "sdxl-inpaint"],
    "formatting": ["ben2", "birefnet", "sdxl-outpaint", "realesrgan", "plain-code"],
    "open_ended": [],
}
SHARED = ["instruct-pix2pix", "magicbrush", "realedit", "vibe", "flux2-klein-4b", "varedit-2b", "flux3-klein",
          "omnigen2"]
TASKS = list(SPECIALISTS)
MODELS = SPECIALISTS if SESSION == "specialists" else {task: SHARED for task in TASKS}
# models that have not passed their smoke run yet: 5 samples per task first (MODEL_LIST.md); remove them from this
# set after the coordinator has checked the smoke run, then the next session runs them on all 30 samples
SMOKE_MODELS = {"realesrgan", "hvi-cidnet", "instructir", "vosr", "instantir", "codeformer", "gfpgan", "ddcolor",
                "ben2", "birefnet", "sdxl-outpaint", "plain-code", "instruct-pix2pix", "magicbrush", "realedit",
                "vibe", "flux2-klein-4b", "varedit-2b", "flux3-klein", "omnigen2"}
# Hugging Face repositories of each model, deleted from the cache after its run (the disk holds about 70 GB)
WEIGHTS = {"rorem": ["LetsThink/RORem"], "objectclear": ["jixin0101/ObjectClear"],
           "osor": ["QinmingZhou/OSOR"],         # its SDXL-Inpainting base is shared with sdxl-inpaint: kept
           "paint-by-inpaint": ["paint-by-inpaint/add-finetuned-mb"], "erasedraw": ["alpercanberk/erasedraw-v0.2"],
           "hvi-cidnet": ["Fediory/HVI-CIDNet-Generalization"], "instructir": ["marcosv/InstructIR"],
           "vosr": [],                            # downloaded into the repository folder in /tmp/models
           "instantir": ["InstantX/InstantIR", "stabilityai/stable-diffusion-xl-base-1.0", "facebook/dinov2-large"],
           "ddcolor": ["piddnad/ddcolor_modelscope"], "ben2": ["PramaLLC/BEN2"], "birefnet": ["ZhengPeng7/BiRefNet"],
           "instruct-pix2pix": ["timbrooks/instruct-pix2pix"], "magicbrush": ["vinesmsuic/magicbrush-jul7"],
           "realedit": ["peter-sushko/RealEdit"], "vibe": ["iitolstykh/VIBE-Image-Edit"],
           "flux2-klein-4b": ["black-forest-labs/FLUX.2-klein-4B"],
           "flux3-klein": ["black-forest-labs/flux-3-image-klein"], "omnigen2": ["OmniGen2/OmniGen2"]}
PLANNED = ["remove", "add_replace", "change_color", "formatting"]       # tasks with a plan (Qwen2.5-VL-7B)
MASKED = ["remove", "add_replace", "change_color"]                    # tasks with masks

In [ ]:
# 1-3. samples (fixed by the dataset: the same every time; in this order, so a sample stays in one task),
# plans (Qwen over both GPUs) and masks; finished ones are skipped
for task in TASKS:
    run_editors("samples", "--task", task)
for task in PLANNED:
    run_editors("plan", "--task", task)
drop_cache("Qwen/Qwen2.5-VL-7B-Instruct")
for task in MASKED:
    run_editors("masks", "--task", task)
drop_cache("IDEA-Research/grounding-dino-base", "facebook/sam2.1-hiera-large")

In [ ]:
# 4. every model, each in its own process; a model that fails does not stop the others
failed = []
order = sorted({model for models in MODELS.values() for model in models},
               key=lambda model: (SHARED.index(model) if model in SHARED else -1, model))
for model in order:                       # one model over all its tasks, then its weights are deleted
    for task in TASKS:
        if model not in MODELS[task]:
            continue
        limit = ["--limit", 5] if model in SMOKE_MODELS else []
        if run_editors("runall", "--task", task, "--models", model, *limit, check=False):
            failed.append(f"{model} ({task})")
    drop_cache(*WEIGHTS.get(model, []))
print("failed:", failed or "none")

In [ ]:
# 5. scores, reports and the summary of every task
from IPython.display import Image, Markdown, display
for task in TASKS:
    if glob.glob(f"{OUT}/scorer/runs/mlp-siglip/head.pt"):
        run_editors("score", "--task", task, check=False)
    run_editors("report", "--task", task)
    sheets = sorted(glob.glob(f"{OUT}/editors/{task}/sheets/*.jpg"))
    if sheets:
        display(Image(sheets[0]))
run_editors("summary")
display(Markdown(open(f"{OUT}/editors/summary.md").read()))

In [ ]:
# Last cell: the clone of the code is removed (the saved output keeps only results), and the whole out/ folder
# goes into one zip next to it: download /kaggle/working/out.zip. The output shows real people: team only.
os.chdir("/kaggle/working")
shutil.rmtree(CODE, ignore_errors=True)
archive = shutil.make_archive("/kaggle/working/out", "zip", "/kaggle/working", "out")
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")